# Building RAG: Chunking, HyDE and Caching

The full code of [Building RAG: Chunking, HyDE and Caching](https://residual-stream.lvtan71.workers.dev/posts/building-rag-chunking-hyde-and-caching/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/building-rag-chunking-hyde-and-caching) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/building-rag-chunking-hyde-and-caching.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 21 min end to end on the RTX 3090 machine that saved the outputs below.
Downloads Qwen2.5-3B-Instruct (about 6 GB) and two small encoders. The post's run also read a draft of Prompt Caching that was in the working tree at the time; this rebuild has 66 posts where the post had 67, so the counts and hits differ by a few. The question set is a regeneration (the post's file is lost), so the counts that depend on it differ by a few.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 21 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "torch==2.14.*" "transformers==5.18.*" huggingface_hub sentence-transformers numpy

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "building-rag-chunking-hyde-and-caching"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/building-rag-chunking-hyde-and-caching


In [2]:
!tar xzf blog-1e98683.tar.gz

## `make_questions.py`

The test set: 150 sentences drawn from the blog's posts, and for each, Qwen2.5-3B-Instruct writes a question it answers, two rewordings and a hypothetical answer. Writes `gen_new.json`, so that it doesn't overwrite the linked `gen.json`.

In [3]:
!{sys.executable} make_questions.py

{"post": "pytorch-from-the-inside", "sentence": "That's post 4.2's lesson from the other side: for memory-bound work, fewer passes is the whole game, until the work stops being memory-bound.", "questi


{"post": "vector-search-and-ann-indexes", "sentence": "Done exactly, it is one line of linear algebra and costs time in proportion to the number of vectors.", "question": "How does the computational c


{"post": "many-agents-subagents-and-orchestration", "sentence": "Take one research question over this blog that needs six posts: how do the inference posts trade memory bandwidth against compute?", "q


{"post": "serving-metrics-and-disaggregation", "sentence": "Near capacity, waiting time grows sharply, as in any queue: time to first token can be fine at 70% of capacity and explode at 95%.", "questi


{"post": "residual-stream-explained", "sentence": "Models pack more features into the stream than it has dimensions, at the cost of some interference between them (Elhage et al., 2022, call this super


...

{"post": "learned-orchestration-sakana-fugu", "sentence": "It is trained with GRPO (post 31): sample a group of workflows for each training question, execute them, and reward each relative to the grou


{"post": "streaming-tokens-sse-websockets", "sentence": "Backpressure is the other direction of the same problem: the client is there but slow, a phone on a poor network.", "question": "How does backp


{"post": "voice-agent-system-design", "sentence": "An iteration takes about 6.5 ms, so at 50 requests a second, each decoding 60 tokens, only about 20 requests are decoding at any moment.", "question"


{"post": "paged-attention-explained", "sentence": "The output is bit-identical to contiguous attention \u2014 paging changes the memory layout, not the computation.", "question": "How does paging diff


{"post": "rope-and-positional-encoding", "sentence": "Take the attention of the attention post, with no position information at all, and permute its input tokens.", "question": "What operation is perf


## `retrieval.py`

Reads `gen.json`. Nine ways of chunking the posts, scored by dense retrieval with MiniLM (hits in the top 1 and top 5, words returned, sign tests); HyDE in four forms; an embedding cache; and a semantic cache at six thresholds, on rewordings, same-post pairs and twelve hand-written near-miss pairs.

In [4]:
!{sys.executable} retrieval.py

66 posts, 150 questions; MiniLM, dense retrieval

chunking                              chunks  words each  findable  hit@1  hit@5  words returned (top 5)  top 5 against recursive 200


fixed, 200 words                         863         192       126   55/150  81/150                     977  


fixed, 200 words, 50 overlap           1,115         195       150   52/150  93/150                     987  


recursive, up to 200 words             1,053         157       150   55/150  96/150                     790  


recursive, up to 100 words             2,049          81       150   71/150 104/150                     410  23 vs 15, p = 0.26


...

  question and answer embeddings averaged hit@1  50  hit@5  92  (vs the question at top 5: 4 gained, 8 lost)



embedding cache: 1,053 chunks embedded in 35.6 s cold, 0.003 s from the cache



semantic cache: 75 questions cached; 300 queries, 150 of them rewordings of a cached question
 threshold  repeats served  wrong answers served  of which to new queries
      0.70         138/150                 1/300                    1/150
      0.75         133/150                 0/300                    0/150
      0.80         120/150                 0/300                    0/150
      0.85         111/150                 0/300                    0/150
      0.90          77/150                 0/300                    0/150
      0.95          37/150                 0/300                    0/150



harder test: 116 pairs of different questions about the same post, against each question's own rewording
 threshold  same-post pairs above it (would be confused)   rewordings above it (hits)
      0.70                                     0/116                      125/150
      0.75                                     0/116                      115/150
      0.80                                     0/116                       93/150
      0.85                                     0/116                       79/150
      0.90                                     0/116                       53/150
      0.95                                     0/116                       24/150
  highest similarity between two different same-post questions: 0.70



hardest test: 12 hand-written pairs that differ by a word or two and need different answers
  similarities: 0.99, 0.98, 0.88, 0.86, 0.79, 0.59, 0.50, 0.48, 0.45, 0.43, 0.42, 0.41
  at threshold 0.80: 4 of 12 pairs would share one cached answer
  at threshold 0.85: 4 of 12 pairs would share one cached answer
  at threshold 0.90: 2 of 12 pairs would share one cached answer
  at threshold 0.95: 2 of 12 pairs would share one cached answer


The post printed:

```text
67 posts, 150 questions; MiniLM, dense retrieval

chunking                              chunks  words each  findable  hit@1  hit@5  words returned (top 5)  top 5 against recursive 200
fixed, 200 words                         870         192       133   52/150  83/150                     976  
fixed, 200 words, 50 overlap           1,124         195       149   51/150  93/150                     986  
recursive, up to 200 words             1,062         157       149   52/150  94/150                     785  
recursive, up to 100 words             2,065          81       149   67/150 103/150                     417  20 vs 11, p = 0.15
one chunk per section                    746         222       149   53/150  85/150                   1,137  16 vs 25, p = 0.21
semantic, up to 200 words              1,577         106       149   67/150 100/150                     607  21 vs 15, p = 0.41
recursive within each section          1,310         124       149   61/150 100/150                     658  15 vs 9, p = 0.31
contextual: title + heading added      1,310         124       149   59/150  98/150                     605  14 vs 10, p = 0.54
small-to-big: 60 words -> section      3,149         304       149   81/150 113/150                   1,267  27 vs 8, p = 0.00
fixed windows, overlap against none: 20 vs 10, p = 0.10; recursive 200 against fixed: 23 vs 12, p = 0.09
recursive 100 at top 10 (a word budget near small-to-big's): hit 114/150, 830 words returned
(MiniLM reads at most 256 word pieces, about 170 words of this blog; the rest of a longer chunk is not embedded)

HyDE, on recursive chunks: what is embedded as the query
  the question                     hit@1  52  hit@5  94  (vs the question at top 5: 0 gained, 0 lost)
  a hypothetical answer (HyDE)     hit@1  31  hit@5  73  (vs the question at top 5: 5 gained, 26 lost)
  question + answer, texts joined  hit@1  40  hit@5  82  (vs the question at top 5: 4 gained, 16 lost)
  question and answer embeddings averaged hit@1  43  hit@5  89  (vs the question at top 5: 3 gained, 8 lost)

embedding cache: 1,062 chunks embedded in 7.8 s cold, 0.001 s from the cache

semantic cache: 75 questions cached; 300 queries, 150 of them rewordings of a cached question
 threshold  repeats served  wrong answers served  of which to new queries
      0.70         137/150                 0/300                    0/150
      0.75         129/150                 0/300                    0/150
      0.80         115/150                 0/300                    0/150
      0.85          95/150                 0/300                    0/150
      0.90          72/150                 0/300                    0/150
      0.95          35/150                 0/300                    0/150

harder test: 114 pairs of different questions about the same post, against each question's own rewording
 threshold  same-post pairs above it (would be confused)   rewordings above it (hits)
      0.70                                     0/114                      125/150
      0.75                                     0/114                      115/150
      0.80                                     0/114                       97/150
      0.85                                     0/114                       79/150
      0.90                                     0/114                       59/150
      0.95                                     0/114                       32/150
  highest similarity between two different same-post questions: 0.66

hardest test: 12 hand-written pairs that differ by a word or two and need different answers
  similarities: 0.99, 0.98, 0.88, 0.86, 0.79, 0.59, 0.50, 0.48, 0.45, 0.43, 0.42, 0.41
  at threshold 0.80: 4 of 12 pairs would share one cached answer
  at threshold 0.85: 4 of 12 pairs would share one cached answer
  at threshold 0.90: 2 of 12 pairs would share one cached answer
  at threshold 0.95: 2 of 12 pairs would share one cached answer
```